In [ ]:
import numpy as np
import pathlib
import tensorflow as tf
from tensorflow.keras.losses import SparseCategoricalCrossentropy

In [ ]:
data_dir = pathlib.Path('/kaggle/input/datasets/vitaliyakavenka/trials-score-training-data')
image_count = len(list(data_dir.glob('*/*.png')))
print(f"Total images: {image_count}")

In [ ]:
img_height = 66
img_width = 640
batch_size = 32

In [ ]:
train_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="training",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=123,
    image_size=(img_height, img_width),
    batch_size=batch_size
)

In [ ]:
normalization_layer = tf.keras.Sequential([
    tf.keras.layers.Rescaling(1./255),
    tf.keras.layers.Lambda(lambda x: tf.image.rgb_to_grayscale(x))
])

normalized_ds = train_ds.map(lambda x, y: (normalization_layer(x), y))
normalized_ds = normalized_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

normalized_val_ds = val_ds.map(lambda x, y: (normalization_layer(x), y))
normalized_val_ds = normalized_val_ds.cache().prefetch(buffer_size=tf.data.AUTOTUNE)

In [ ]:
model = tf.keras.Sequential([
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.MaxPooling2D(),
    tf.keras.layers.Conv2D(32, 3, activation='relu'),
    tf.keras.layers.Flatten(),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dense(5)
])

In [ ]:
model.compile(
    optimizer='adam',
    loss=SparseCategoricalCrossentropy(from_logits=True),
    metrics=['accuracy']
)

history = model.fit(
    normalized_ds,
    validation_data=normalized_val_ds,
    epochs=10
)

In [ ]:
final_val_acc = history.history['val_accuracy'][-1]
assert final_val_acc >= 0.85, f"Validation accuracy {final_val_acc:.2%} < 85% threshold"
print(f"Validation accuracy: {final_val_acc:.2%} ✓")

In [ ]:
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS
]
tflite_model = converter.convert()
with open('/kaggle/working/score_classifier_model.tflite', 'wb') as f:
    f.write(tflite_model)
print(f"Model saved: {len(tflite_model)/1024:.1f} KB")